# EMT 3-phase RLC State-Space Extraction

This notebook validates MNA state-space extraction for a balanced EMT three-phase RLC circuit.

Three equivalent model variants are considered:

1. **Native EMT MNA components**: the circuit is built from standard EMT MNA components.
2. **SSN load**: the RLC load is replaced by a generic two-terminal V-type SSN component.
3. **SSN branch**: the series RL branch is replaced by a generic two-terminal V-type SSN component.

For each case, the notebook extracts the discrete-time state matrix, computes modal quantities using `StateSpaceModalAnalysis`, and checks the extracted eigenvalues against the analytical reference.

In [ ]:
import numpy as np
import dpsimpy

emt = dpsimpy.emt
ph3 = emt.ph3

Simulation = dpsimpy.Simulation
SystemTopology = dpsimpy.SystemTopology
Domain = dpsimpy.Domain
Solver = dpsimpy.Solver
PhaseType = dpsimpy.PhaseType
StateSpaceModalAnalysis = dpsimpy.StateSpaceModalAnalysis

## Parameters

In [ ]:
time_step = 1e-4
final_time = 1e-3
frequency = 50.0

source_voltage = 1.0 * np.exp(1j * 0.0)

i3 = np.eye(3)

branch_resistance = 0.2 * i3
branch_inductance = 0.02 * i3

load_resistance = 1.0 * i3
load_inductance = 0.05 * i3
load_capacitance = 100e-6 * i3

## Helper functions

In [ ]:
def abc(phasor):
    return np.array(
        [
            [phasor],
            [phasor * np.exp(-1j * 2.0 * np.pi / 3.0)],
            [phasor * np.exp(+1j * 2.0 * np.pi / 3.0)],
        ],
        dtype=np.complex128,
    )


def print_complex_vector(values, indent="  "):
    for idx, value in enumerate(values):
        print(f"{indent}[{idx}] {value}")


def create_rlc_load_ssn_matrices():
    inv_l = np.linalg.inv(load_inductance)
    inv_c = np.linalg.inv(load_capacitance)

    # x = [uC_abc; i_abc], input = v_abc, output = i_abc
    a = np.zeros((6, 6))
    a[0:3, 3:6] = inv_c
    a[3:6, 0:3] = -inv_l
    a[3:6, 3:6] = -inv_l @ load_resistance

    b = np.zeros((6, 3))
    b[3:6, 0:3] = inv_l

    c = np.zeros((3, 6))
    c[0:3, 3:6] = i3

    d = np.zeros((3, 3))
    return a, b, c, d


def create_rl_branch_ssn_matrices():
    inv_l = np.linalg.inv(branch_inductance)

    # x = i_abc, input = v_abc, output = i_abc
    a = -inv_l @ branch_resistance
    b = inv_l
    c = i3.copy()
    d = np.zeros((3, 3))
    return a, b, c, d


def map_continuous_to_discrete(lam):
    return (2.0 + time_step * lam) / (2.0 - time_step * lam)


def expected_modes():
    total_resistance = branch_resistance[0, 0] + load_resistance[0, 0]
    total_inductance = branch_inductance[0, 0] + load_inductance[0, 0]
    capacitance = load_capacitance[0, 0]

    alpha = total_resistance / (2.0 * total_inductance)
    omega0 = np.sqrt(1.0 / (total_inductance * capacitance))
    omega_d = np.sqrt(omega0**2 - alpha**2)

    lambda_expected = np.array(
        [
            -alpha + 1j * omega_d,
            -alpha - 1j * omega_d,
        ],
        dtype=np.complex128,
    )

    z_expected = map_continuous_to_discrete(lambda_expected)

    return lambda_expected, z_expected


lambda_expected, z_expected = expected_modes()

print("Expected continuous-time modes per phase:")
print(lambda_expected)

print("\nExpected trapezoidal discrete-time modes per phase:")
print(z_expected)


def configure_and_run(system, sim_name):
    sim = Simulation(sim_name)
    sim.set_system(system)
    sim.set_domain(Domain.EMT)
    sim.set_solver(Solver.MNA)
    sim.set_time_step(time_step)
    sim.set_final_time(final_time)
    sim.do_state_space_extraction(True)
    sim.run()
    return sim


def get_modal_results(sim):
    extractor = sim.get_state_space_extractor()

    modal_analysis = StateSpaceModalAnalysis(extractor)
    modal_analysis.update()

    ad = np.array(extractor.get_discrete_state_matrix(), copy=True)
    z = np.array(modal_analysis.get_discrete_eigenvalues(), copy=True).reshape(-1)
    lambdas = np.array(modal_analysis.get_continuous_eigenvalues(), copy=True).reshape(
        -1
    )

    return {
        "state_count": extractor.get_state_count(),
        "Ad": ad,
        "z": z,
        "lambda": lambdas,
    }

## Case 1: native EMT MNA components

In [ ]:
def run_native_components_case():
    n1 = emt.SimNode("n1", PhaseType.ABC)
    n2 = emt.SimNode("n2", PhaseType.ABC)
    n3 = emt.SimNode("n3", PhaseType.ABC)
    n4 = emt.SimNode("n4", PhaseType.ABC)
    n5 = emt.SimNode("n5", PhaseType.ABC)

    vs = ph3.VoltageSource("VS")
    vs.set_parameters(abc(source_voltage), frequency)

    r_branch = ph3.Resistor("RBranch")
    r_branch.set_parameters(branch_resistance)

    l_branch = ph3.Inductor("LBranch")
    l_branch.set_parameters(branch_inductance)

    r_load = ph3.Resistor("RLoad")
    r_load.set_parameters(load_resistance)

    l_load = ph3.Inductor("LLoad")
    l_load.set_parameters(load_inductance)

    c_load = ph3.Capacitor("CLoad")
    c_load.set_parameters(load_capacitance)

    vs.connect([emt.SimNode.gnd, n1])
    r_branch.connect([n1, n2])
    l_branch.connect([n2, n3])
    r_load.connect([n3, n4])
    l_load.connect([n4, n5])
    c_load.connect([n5, emt.SimNode.gnd])

    system = SystemTopology(
        frequency,
        [n1, n2, n3, n4, n5],
        [vs, r_branch, l_branch, r_load, l_load, c_load],
    )

    return configure_and_run(system, "EMT_Ph3_RLC_StateSpaceExtraction_Native")

## Case 2: native RL branch and SSN RLC load

In [ ]:
def run_native_branch_ssn_load_case():
    n1 = emt.SimNode("n1", PhaseType.ABC)
    n2 = emt.SimNode("n2", PhaseType.ABC)
    n3 = emt.SimNode("n3", PhaseType.ABC)

    a, b, c, d = create_rlc_load_ssn_matrices()

    vs = ph3.VoltageSource("VS")
    vs.set_parameters(abc(source_voltage), frequency)

    r_branch = ph3.Resistor("RBranch")
    r_branch.set_parameters(branch_resistance)

    l_branch = ph3.Inductor("LBranch")
    l_branch.set_parameters(branch_inductance)

    rlc_load = ph3.GenericTwoTerminalVTypeSSN("RLCLoad")
    rlc_load.set_parameters(a, b, c, d)

    vs.connect([emt.SimNode.gnd, n1])
    r_branch.connect([n1, n2])
    l_branch.connect([n2, n3])
    rlc_load.connect([n3, emt.SimNode.gnd])

    system = SystemTopology(
        frequency,
        [n1, n2, n3],
        [vs, r_branch, l_branch, rlc_load],
    )

    return configure_and_run(system, "EMT_Ph3_RLC_StateSpaceExtraction_SSNLoad")

## Case 3: SSN RL branch and native RLC load

In [ ]:
def run_ssn_branch_native_load_case():
    n1 = emt.SimNode("n1", PhaseType.ABC)
    n3 = emt.SimNode("n3", PhaseType.ABC)
    n4 = emt.SimNode("n4", PhaseType.ABC)
    n5 = emt.SimNode("n5", PhaseType.ABC)

    a, b, c, d = create_rl_branch_ssn_matrices()

    vs = ph3.VoltageSource("VS")
    vs.set_parameters(abc(source_voltage), frequency)

    rl_branch = ph3.GenericTwoTerminalVTypeSSN("RLBranch")
    rl_branch.set_parameters(a, b, c, d)

    r_load = ph3.Resistor("RLoad")
    r_load.set_parameters(load_resistance)

    l_load = ph3.Inductor("LLoad")
    l_load.set_parameters(load_inductance)

    c_load = ph3.Capacitor("CLoad")
    c_load.set_parameters(load_capacitance)

    vs.connect([emt.SimNode.gnd, n1])
    rl_branch.connect([n1, n3])
    r_load.connect([n3, n4])
    l_load.connect([n4, n5])
    c_load.connect([n5, emt.SimNode.gnd])

    system = SystemTopology(
        frequency,
        [n1, n3, n4, n5],
        [vs, rl_branch, r_load, l_load, c_load],
    )

    return configure_and_run(system, "EMT_Ph3_RLC_StateSpaceExtraction_SSNBranch")

## Assertions

The balanced three-phase circuit should have 9 extracted states: three phases times three discrete extraction states per phase. The physical finite continuous-time modes should match the analytical RLC reference, and the redundant trapezoidal mode should appear as `z = -1` once per phase.

In [ ]:
def sort_complex(values):
    return np.array(
        sorted(values, key=lambda value: (round(value.imag, 8), round(value.real, 8)))
    )


def assert_modal_results(case_name, results):
    state_count = results["state_count"]
    z = results["z"]
    lambdas = results["lambda"]

    assert state_count == 9, f"{case_name}: expected 9 states, got {state_count}"
    assert len(z) == 9, f"{case_name}: expected 9 discrete eigenvalues, got {len(z)}"
    assert (
        len(lambdas) == 9
    ), f"{case_name}: expected 9 continuous eigenvalues, got {len(lambdas)}"

    redundant_modes = np.isclose(z, -1.0 + 0.0j, atol=1e-10)
    assert np.count_nonzero(redundant_modes) == 3, (
        f"{case_name}: expected three redundant z=-1 modes, "
        f"got {np.count_nonzero(redundant_modes)}"
    )

    finite_lambdas = lambdas[np.isfinite(lambdas.real) & np.isfinite(lambdas.imag)]

    assert len(finite_lambdas) == 6, (
        f"{case_name}: expected six finite continuous-time modes, "
        f"got {len(finite_lambdas)}"
    )

    expected_repeated = np.repeat(lambda_expected, 3)

    np.testing.assert_allclose(
        sort_complex(finite_lambdas),
        sort_complex(expected_repeated),
        rtol=1e-6,
        atol=1e-6,
        err_msg=f"{case_name}: finite continuous-time modes do not match",
    )

## Run validation

The three model variants are simulated and checked against the same analytical reference.

In [ ]:
cases = {
    "native_components": run_native_components_case(),
    "native_branch_ssn_load": run_native_branch_ssn_load_case(),
    "ssn_branch_native_load": run_ssn_branch_native_load_case(),
}

for case_name, sim in cases.items():
    results = get_modal_results(sim)

    print(f"\n{case_name}")
    print("state count:", results["state_count"])
    print("discrete eigenvalues z:")
    print_complex_vector(results["z"])
    print("continuous eigenvalues lambda:")
    print_complex_vector(results["lambda"])

    assert_modal_results(case_name, results)

print("\nAll state-space extraction and modal-analysis checks passed.")